# NOVA 2.0 — Improved training notebook

This revision uses 50 reviewed starter examples and masks the prompt tokens so the loss is calculated on assistant responses only. It is still a small experiment, not a complete self-learning system. Select Runtime → Change runtime type → T4 GPU if available.

In [ ]:
!nvidia-smi || true
!pip -q install transformers==4.48.3 datasets peft accelerate bitsandbytes gradio


In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
REPO = 'https://raw.githubusercontent.com/barmanbiswa131-design/My-app-bb/nova2-prototype/training/data/train_v2.jsonl'
!wget -q -O /content/nova_train.jsonl "$REPO"
rows = [json.loads(x) for x in open('/content/nova_train.jsonl', encoding='utf-8') if x.strip()]
print('Training examples:', len(rows), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
assert torch.cuda.is_available(), 'T4 GPU নির্বাচন করে Runtime পুনরায় চালাও।'
BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
SYSTEM = 'You are NOVA, an AI assistant created by Biswajit Barman. Your name is NOVA. Understand Bengali, Banglish, Hindi and Hinglish. Reply in the same language as the user. Give a direct, relevant, natural answer. Never invent unrelated numbered lists.'


In [ ]:
def encode_example(item):
    prompt_messages = [{'role':'system','content':SYSTEM}, {'role':'user','content':item['user']}]
    full_messages = prompt_messages + [{'role':'assistant','content':item['assistant']}]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    full = tokenizer(full_text, truncation=True, max_length=384, add_special_tokens=False)
    labels = full['input_ids'].copy()
    labels[:min(len(prompt_ids), len(labels))] = [-100] * min(len(prompt_ids), len(labels))
    full['labels'] = labels
    return full
encoded = [encode_example(x) for x in rows]
ds = Dataset.from_list(encoded)
print(ds)
print('Assistant-only loss masking ready.')


In [ ]:
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=quant, device_map='auto')
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
lora = LoraConfig(task_type='CAUSAL_LM', r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none')
model = get_peft_model(model, lora)
model.print_trainable_parameters()
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100, return_tensors='pt')
args = TrainingArguments(output_dir='/content/nova2-checkpoints', num_train_epochs=4, per_device_train_batch_size=1, gradient_accumulation_steps=4, learning_rate=1e-4, logging_steps=5, save_strategy='epoch', fp16=True, report_to='none', remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()
ADAPTER='/content/nova2-improved-adapter'
model.save_pretrained(ADAPTER); tokenizer.save_pretrained(ADAPTER)
print('Improved adapter saved:', ADAPTER)


In [ ]:
def ask_nova(prompt):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':prompt}]
    rendered=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(rendered, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out=model.generate(**inputs, max_new_tokens=100, do_sample=False, repetition_penalty=1.08)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

tests=['তোমার নাম কী?','তোমাকে কে তৈরি করেছে?','তুমি কেমন আছো?','Tomar nam ki?','Tumhara naam kya hai?','तुम्हारा नाम क्या है?']
for q in tests:
    print('USER:', q, '\\nNOVA:', ask_nova(q), '\\n')


## Save the adapter to Google Drive

Run this cell after training. It stores a ZIP copy in your Drive so the adapter is not lost when the Colab session ends.

In [ ]:
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
zip_path = shutil.make_archive('/content/NOVA_2_0_Improved_Adapter', 'zip', ADAPTER)
shutil.copy2(zip_path, '/content/drive/MyDrive/NOVA_2_0_Improved_Adapter.zip')
print('Saved to Google Drive: MyDrive/NOVA_2_0_Improved_Adapter.zip')


In [ ]:
import gradio as gr
demo = gr.Interface(fn=ask_nova, inputs=gr.Textbox(lines=2, label='Message to NOVA'), outputs=gr.Textbox(label='NOVA reply'), title='NOVA 2.0 — Improved experiment')
demo.launch(share=True)
